# Phase A — Detection MVP on Climbing Holds Dataset

Trains `v9-t` and `v9-s` (detection only) on the 19-image hand-labelled climbing dataset.

**Prereqs:**
1. Drive folder `My Drive/climbing-holds/` exists with `data/climbing_holds.zip` uploaded
2. Fork at `https://github.com/ob-choco/YOLO` has the `feature/climbing-seg` branch pushed

**Output:** checkpoints + visualizations to `My Drive/climbing-holds/runs/phaseA-<size>-<datetime>/`

**Runtime:** Colab Free T4 → ~1–3 hours per model size.

**Spec / plan refs:**
- `docs/superpowers/specs/2026-05-06-climbing-holds-segmentation-design.md`
- `docs/superpowers/plans/2026-05-06-climbing-holds-segmentation.md` Tasks 7, 8

## 1. Mount Drive and clone the fork

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
os.chdir("/content")
if not os.path.exists("YOLO"):
    subprocess.check_call([
        "git", "clone", "-b", "feature/climbing-seg",
        "https://github.com/ob-choco/YOLO.git",
    ])
os.chdir("/content/YOLO")
subprocess.check_call(["git", "pull"])
print(subprocess.check_output(["git", "log", "--oneline", "-3"]).decode())

## 2. Install dependencies

Colab pre-installs torch with CUDA; the rest of `requirements.txt` covers Lightning/Hydra/PyCOCO/etc.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "numpy<2" pycocotools  # numpy<2 keeps torch interop happy on Colab too
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 3. Unzip dataset from Drive

Skip if already unzipped (Colab session was reused).

In [ ]:
import os
if not os.path.exists("/content/YOLO/data/climbing_holds/annotations/instances_train.json"):
    !mkdir -p /content/YOLO/data
    !unzip -q -o /content/drive/MyDrive/climbing-holds/data/climbing_holds.zip -d /content/YOLO/
!ls /content/YOLO/data/climbing_holds/images/train | wc -l   # expect 1061
!ls /content/YOLO/data/climbing_holds/images/val   | wc -l   # expect 1212
!ls /content/YOLO/data/climbing_holds/annotations/  # expect instances_train.json, instances_val.json

## 4. Download pretrained weights (v9-t and v9-s detection)

From the upstream MultimediaTechLab/YOLO release. Cached on Drive after the first run.

In [ ]:
import os, urllib.request, shutil
os.makedirs("weights", exist_ok=True)
drive_pretrained = "/content/drive/MyDrive/climbing-holds/pretrained"
os.makedirs(drive_pretrained, exist_ok=True)

for ckpt in ["v9-t.ckpt", "v9-s.ckpt"]:
    drive_path = os.path.join(drive_pretrained, ckpt)
    local_path = os.path.join("weights", ckpt)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
        print(f"Loaded {ckpt} from Drive cache")
        continue
    url = f"https://github.com/MultimediaTechLab/YOLO/releases/download/v1.0-alpha/{ckpt}"
    try:
        urllib.request.urlretrieve(url, local_path)
        shutil.copy(local_path, drive_path)
        print(f"Downloaded {ckpt} ({os.path.getsize(local_path)} bytes), cached to Drive")
    except Exception as e:
        print(f"⚠️  Failed to download {ckpt}: {e}")
        print(f"    Training can still proceed with weight=False (random init), just slower convergence.")

## 5a. Train v9-t (detection)

50 epochs, batch=8, image_size=640, T4 GPU.

Override flags learned during the laptop smoke test:
- `accelerator=gpu device=1` (Colab T4)
- `image_size=[640,640]` (list, not int)
- `use_wandb=False` (no API key on Colab)

In [ ]:
import datetime, subprocess
run_name = f"phaseA-v9-t-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg = "weights/v9-t.ckpt" if os.path.exists("weights/v9-t.ckpt") else "False"
subprocess.check_call([
    "python", "yolo/lazy.py",
    "task=train",
    "model=v9-t",
    "dataset=climbing_holds",
    "task.data.batch_size=8",
    "image_size=[640,640]",
    "task.epoch=50",
    "accelerator=gpu",
    "device=1",
    "use_wandb=False",
    f"weight={weight_arg}",
    f"name={run_name}",
])
print("v9-t run:", run_name)

## 5b. Train v9-s (detection)

In [ ]:
run_name_s = f"phaseA-v9-s-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg_s = "weights/v9-s.ckpt" if os.path.exists("weights/v9-s.ckpt") else "False"
subprocess.check_call([
    "python", "yolo/lazy.py",
    "task=train",
    "model=v9-s",
    "dataset=climbing_holds",
    "task.data.batch_size=8",
    "image_size=[640,640]",
    "task.epoch=50",
    "accelerator=gpu",
    "device=1",
    "use_wandb=False",
    f"weight={weight_arg_s}",
    f"name={run_name_s}",
])
print("v9-s run:", run_name_s)

## 6. Save artifacts to Drive

Each run directory under `runs/train/<name>/` contains: `last.ckpt`, `best.ckpt` (if checkpoints fired), `val_pred_epoch*.png`, hydra logs.

In [ ]:
import shutil
for run in (run_name, run_name_s):
    src = f"/content/YOLO/runs/train/{run}"
    dst = f"/content/drive/MyDrive/climbing-holds/runs/{run}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Saved:", dst)
    else:
        print("Run dir missing (training failed?):", src)

## 7. Quick visual / metrics review

Verify val PNG overlays look sensible before moving to Phase B.

In [ ]:
from IPython.display import Image as _Img, display
import glob, pandas as pd
for run in (run_name, run_name_s):
    pngs = sorted(glob.glob(f"runs/train/{run}/val_pred_epoch*.png"))
    if pngs:
        print(f"\n=== {run} (latest val PNG) ===")
        display(_Img(pngs[-1]))
    csvs = glob.glob(f"runs/train/{run}/**/metrics.csv", recursive=True)
    if csvs:
        print("metrics tail:", pd.read_csv(csvs[0]).tail(5))